In [1]:
import numpy as np
import functions as fn
from circuit_obj import Circuit
from tqdm import tqdm
from itertools import product
from matplotlib import pyplot as plt
import json

In [34]:
# Define your parameter arrays
N = 6
T = 1000
L = 5 # How fine the grid: L x L
circuit_realizations = 10

masks_dict = fn.load_mask_memory(N, 2)

In [ ]:
# load the file /Users/a.summer/Downloads/random_pauli-main/combined_results/entanglement/N14/T1000/Jx0.10/Jz0.10/combined.npz

npz_file = np.load('/Users/a.summer/Downloads/random_pauli-main/combined_results/entanglement/N12/T1000/Jx0.10/Jz0.10/combined.npz')

# extract the data without knowing the keys
keys = npz_file.files
data = {key: npz_file[key] for key in keys}
# print the keys and their shapes
for key, value in data.items():
    print(f"{key}: {value.shape}")

In [ ]:
# Function to load individual circuit realization files into matrices
def load_individual_files(base_dir, observable, N, T, L, circuit_realizations):
    """
    Load all individual cr{CR}.npz files into matrices of shape (circuit_realizations, L, L, T+1)
    
    Parameters:
    -----------
    base_dir : str
        Base directory containing the results
    observable : str
        'magic' or 'entanglement'
    N : int
        Number of qubits
    T : int
        Number of timesteps
    L : int
        Grid size (L x L)
    circuit_realizations : int
        Number of circuit realizations
        
    Returns:
    --------
    data_matrix : numpy.ndarray
        Matrix of shape (circuit_realizations, L, L, T+1) containing the data
    """
    data_matrix = np.zeros((circuit_realizations, L, L, T+1))
    
    # Define the parameter grid
    jx_values = np.linspace(0.1, np.pi-0.1, L)
    jz_values = np.linspace(0.1, np.pi-0.1, L)
    
    for cr in range(circuit_realizations):
        for i, Jx in enumerate(jx_values):
            for j, Jz in enumerate(jz_values):
                # Construct the path to the individual file
                file_path = f"{base_dir}/{observable}/N{N}/T{T}/Jx{Jx:.2f}/Jz{Jz:.2f}/cr{cr}.npz"
                
                try:
                    data = np.load(file_path)
                    
                    # Handle both possible keys for entanglement
                    if observable == 'magic':
                        key = 'magic'
                    elif observable == 'entanglement':
                        if 'entan' in data:
                            key = 'entan'
                        elif 'entanglement' in data:
                            key = 'entanglement'
                        else:
                            print(f"Warning: No valid key found in {file_path}")
                            continue
                    
                    # Store the data in our matrix
                    data_matrix[cr, i, j, :] = data[key]
                    
                except FileNotFoundError:
                    print(f"File not found: {file_path}")
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    return data_matrix

# Example usage:
base_dir = '/Users/a.summer/Downloads/random_pauli-main/combined_results'

# Load magic and entanglement data
magic_matrix = load_individual_files(base_dir, 'magic', N, T, L, circuit_realizations)
entanglement_matrix = load_individual_files(base_dir, 'entanglement', N, T, L, circuit_realizations)

print(f"Magic matrix shape: {magic_matrix.shape}")
print(f"Entanglement matrix shape: {entanglement_matrix.shape}")

# Calculate averages over circuit realizations (if needed)
magic_avg = np.mean(magic_matrix, axis=0)
entanglement_avg = np.mean(entanglement_matrix, axis=0)

print(f"Magic average shape: {magic_avg.shape}")  # Should be (L, L, T+1)
print(f"Entanglement average shape: {entanglement_avg.shape}")  # Should be (L, L, T+1)

# You can now use these matrices for your phase space diagrams or other analyses

In [ ]:
def initial_state(N, n):
    """
    Generate a random state vector on N qubits with spin up at site n
    """
    up = np.array([0, 1])
    if n == 0:
        rnd_part = np.random.rand(2**(N - 1)) + 1j * np.random.rand(2**(N - 1))
        state = np.kron(up, rnd_part)
        state /= np.linalg.norm(state)
        return state
    elif n == N - 1:
        rnd_part = np.random.rand(2**(N - 1)) + 1j * np.random.rand(2**(N - 1))
        state = np.kron(rnd_part, up)
        state /= np.linalg.norm(state)
        return state
    part1 = np.random.rand(2**(n)) + 1j * np.random.rand(2**(n))
    part2 = np.random.rand(2**(N - n - 1)) + 1j * np.random.rand(2**(N - n - 1))
    state = np.kron(part1, np.kron(up, part2))
    state /= np.linalg.norm(state)
    return state

# sanity check :
# [np.round(fn.get_magnetization(initial_state(N, idx), N)[idx]) for idx in range(N)]

st = initial_state(N, 0)

In [35]:
with open(f'../random_archive/angles_N{N}.json', 'r', encoding='utf-8') as f:
    loaded_dict = json.load(f)
    
circuits = []

TARGETS = [0,  np.pi/2, -np.pi/2, np.pi]

def quantize_angle(x: float) -> float:
    """
    Map x in [-pi, pi] to the nearest of {0, ±pi/2, pi}.
    """
    # ensure x is in the domain
    if x < -np.pi or x > np.pi:
        raise ValueError("x must be in [-pi, pi]")
    # find the target with minimal absolute difference
    return min(TARGETS, key=lambda s: abs(x - s))

for cr in range(circuit_realizations):
    circuits_Jx = []
    for Jx in np.linspace(.001, np.pi-.001, L):
        circuits_Jz = []
        for Jz in np.linspace(.001, np.pi-.001, L):
            gates = []
            for n in range(N):
                params = loaded_dict[cr][n]
                # Jz = params['Jz']; Jx = params['Jx']
                θ1 = quantize_angle(params['θ1']); θ2 = quantize_angle(params['θ2'])
                θ3 = quantize_angle(params['θ3']); θ4 = quantize_angle(params['θ4'])
                gates.append(fn.gate_xyz_disordered(θ1, θ2, Jx/4, Jx/4, Jz/4, θ3, θ4)) # h1, h2, Jx, Jy, Jz, h3, h4
            
            gates = np.array(gates)
            assert len(gates) == N, f"Expected {N} gates, got {gates.shape}"

            order = fn.gen_gates_order(N)    
            circuit = Circuit(N=N, gates=gates, order=order)
            circuit.couplings = [Jx, Jz]
            circuit.verbose = False
            circuits_Jz.append(circuit)
        circuits_Jx.append(circuits_Jz)
    circuits.append(circuits_Jx)
            
def random_basis_state(n_qubits, seed=None):
    if seed is not None:
        np.random.seed(seed)    
    x = np.random.randint(0, 2**n_qubits)
    psi = np.zeros(2**n_qubits, dtype=complex)
    psi[x] = 1.0
    return psi

##############################################################################################

def compute_correlation(idx, circuit):
    state = initial_state(N, idx) # initial_state_test(theta)
    return circuit.run(masks_dict, state, T, objective=['correlation'])

def compute_magic(circuit):
    state = random_basis_state(N)
    return circuit.run(masks_dict, state, T, objective=['magic'])

def compute_entanglement(circuit):
    state = random_basis_state(N)
    return circuit.run(masks_dict, state, T, objective=['entanglement'])

def compute_magic_entanglement(circuit):
    state = random_basis_state(N)
    return circuit.run(masks_dict, state, T, objective=['magic', 'entanglement'])

In [36]:
if globals().get('correlation') is None or globals(
    ).get('correlation').shape != (N, circuit_realizations * (L**2), T + 1, N):
    correlation = np.zeros((N, circuit_realizations * (L**2), T + 1, N), dtype=np.float64)
    
if globals().get('magic') is None or globals(
    ).get('magic').shape != (circuit_realizations, L, L, T + 1):
    magic = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)
    
if globals().get('entan') is None or globals(
    ).get('entan').shape != (circuit_realizations, L, L, T + 1):
    entan = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)

for idx, Lx, Lz, cr in tqdm(
        product(
            range(1),
            range(L),
            range(L),
            range(circuit_realizations),
        ),
        total=1 * L * L * circuit_realizations,
        desc='Computing magic and entanglement', disable=False):
    circuit = circuits[cr][Lx][Lz]
    output, _ = compute_magic_entanglement(circuit);
    magic[cr, Lx, Lz, :], entan[cr, Lx, Lz, :] = output[0], output[1]

Computing magic and entanglement:   0%|          | 0/250 [00:00<?, ?it/s]

Computing magic and entanglement: 100%|██████████| 250/250 [1:21:12<00:00, 19.49s/it]


In [ ]:
# # np.save('result_N16.npy', result)
# result = np.load('result_N16.npy')
# with open("nathan.txt", "r") as f:
#     lines = f.readlines()                    # read all lines as strings
# data =[float(i) for i in lines[0].split(', ')]  # drop blank lines
# print(data)

In [ ]:
# # save magic in a npz file
# np.savez(
#     f'../random_archive/magic_entanglement_N{N}_T{T}_L{L}_cr{circuit_realizations}.npz',
#     magic=magic,
#     entan=entan,
# )
# load the file
N = 14; T = 1000; L = 3; circuit_realizations = 10
npz_file = np.load(
    f'../random_archive/magic_entanglement_N{N}_T{T}_L{L}_cr{circuit_realizations}.npz')
magic = npz_file['magic']
entan = npz_file['entan']

In [17]:
from scipy.special import comb, gammaln

def avg_renyi_u1_covariant(L: int, L_A: int, alpha: float) -> float:
    """
    Average Rényi-α entropy for the U(1)-covariant ensemble on L qubits,
    each state has support on all total-charge N=0…L, of which the subsystem
    A has L_A qubits.  α != 1.
    
    Implements
      ⟨Tr ρ_A^α⟩ 
        = ∑_{N=0}^{L_A} (d_N)_{(α)} · d_{N,A}^{1-α} 
          / (D)_{(α)},
    where
      d_N    = C(L, N),
      d_{N,A}= C(L_A, N),
      D      = 2^L,
    and (x)_{(α)} = Γ(x+α)/Γ(x).
    Finally ⟨S_α⟩ = (1/(1−α))·ln⟨Tr ρ_A^α⟩.
    """
    if alpha == 1.0:
        raise ValueError("α = 1 is singular; take α→1 limit separately.")
    # total D
    D = 2**L
    # precompute log‐denominator rising Pochhammer
    log_D_poch = gammaln(D + alpha) - gammaln(D)

    acc = 0.0
    for N in range(0, L_A+1):
        dN  = comb(L,  N, exact=True)
        dNA = comb(L_A, N, exact=True)
        # rising Pochhammer for global-sector: (dN)_(α) = Γ(dN+α)/Γ(dN)
        log_num = gammaln(dN + alpha) - gammaln(dN)
        # block-factor: dNA^(1−α)
        log_block = (1.0 - alpha) * np.log(dNA)
        acc += np.exp(log_num + log_block - log_D_poch)

    tr_rho_alpha = acc
    return np.log(tr_rho_alpha) / (1.0 - alpha)

In [18]:
avg_renyi_u1_covariant(6, 3, 2)

2.0871635877737393

In [39]:
%matplotlib osx

fig, axs = plt.subplots(L, L, figsize=(12, 10), sharex=True, sharey=True)
for Lx, Lz in product(range(L), range(L)):
    ax = axs[L-Lx-1, Lz]
    ax.axhline(avg_renyi_u1_covariant(N, N//2, 2), color='black', linestyle='--', label='Average U(1) Renyi-2 Entropy', lw=.75)
    
    Jx = np.linspace(.001, np.pi-.001, L)[Lx]
    Jz = np.linspace(.001, np.pi-.001, L)[Lz]
    ax.plot(magic[:,Lx,Lz,:].mean(axis=(0)), label=f'Magic Jx={Jx:.3f}π, Jz={Jz:.3f}π', marker='.')
    ax.plot(entan[:,Lx,Lz,:].mean(axis=(0)), label=f'Entan Jx={Jx:.3f}π, Jz={Jz:.3f}π', marker='.')
    if Lx == 0:
        ax.set_xlabel('Time')
    if Lz == 0:
        ax.set_ylabel('Magnetization')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend(fontsize=5)
    
fig.suptitle(f'Phase Space Diagram U(1) with N={N}, with discrete noise', fontsize=16) #  #cr={circuit_realizations}
fig.tight_layout()
#set title to the whole figure

In [ ]:
plt.loglog(magic[:,2,0,:].mean(axis=(0)))
magic[:,2,0,:].mean(axis=(0))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from itertools import product
from tqdm import trange

# Reconstruct your time axis (log‐spaced 10^0…10^3 over T points):
T = magic.shape[-1]
time = np.arange(T)

# Grid size:
L = magic.shape[1]

def detect_powerlaw_restricted(x, y, R2_min=0.90, Nmin=10, alpha=0.8):
    """
    Finds the longest high-R2 power-law segment *before* the curve
    reaches alpha*max(y).  Returns (i0, j0, slope, prefactor) or None.
    """
    # 1) cutoff at saturation
    ymax = y.max()
    sat_idx = np.argmax(y >= alpha * ymax)
    if sat_idx < Nmin:
        return None

    x2 = x[:sat_idx]
    y2 = y[:sat_idx]
    mask = (x2 > 0) & (y2 > 0)
    x2, y2 = x2[mask], y2[mask]
    if len(x2) < Nmin:
        return None

    # 2) scan in log space
    logx, logy = np.log(x2), np.log(y2)
    Np = len(logx)
    best = {'length': 0}
    for i in trange(Np):
        for j in range(i + Nmin, Np):
            slope, intercept, r_value, _, _ = linregress(logx[i:j], logy[i:j])
            R2 = r_value**2
            length = j - i
            if R2 >= R2_min and length > best['length']:
                best.update(i=i, j=j, slope=slope, intercept=intercept,
                            length=length)
    if 'i' not in best:
        return None

    # map back to original indices
    idx = np.where(mask)[0]
    i0 = idx[ best['i'] ]
    j0 = idx[ best['j'] - 1 ]
    A = np.exp(best['intercept'])
    return i0, j0, best['slope'], A


# Plot full L×L grid:
fig, axs = plt.subplots(L, L, figsize=(12, 10), sharex=True, sharey=True)
for Lx, Lz in product(range(L), range(L)):
    ax = axs[L-1-Lx, Lz]
    # mean over realizations
    y_magic = magic[:, Lx, Lz, :].mean(axis=0)
    y_entan = entan[:, Lx, Lz, :].mean(axis=0)

    # raw traces
    ax.plot(time, y_magic,  label='Magic', marker='.', ms=3)
    ax.plot(time, y_entan,   label='Entan', marker='.', ms=3)

    # detect & overlay fits
    for y, col, lbl in [(y_magic, 'C0', 'Magic'), (y_entan, 'C1', 'Entan')]:
        res = None
        res = detect_powerlaw_restricted(time, y,
                                         R2_min=0.90,
                                         Nmin=10,
                                         alpha=0.8)
        if res is not None:
            i0, j0, m, A = res
            tx = time[i0:j0+1]
            yfit = A * tx**m
            ax.plot(tx, yfit, '--', color=col, lw=1)
            ax.text(0.05,
                    0.9 - 0.08*(lbl=='Entan'),
                    f'{lbl}: t^{m:.2f}',
                    transform=ax.transAxes,
                    color=col,
                    fontsize=6)

    # force log–log
    ax.set_xscale('log')
    ax.set_yscale('log')

    if Lx == 0:
        ax.set_xlabel('Time')
    if Lz == 0:
        ax.set_ylabel('Magnetization')
    ax.legend(fontsize=5)

fig.suptitle('Phase Space Diagram U(1) – Saturation-Restricted Power-Law Fits',
             fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
fn.print_matrix(fn.gate_xyz_disordered(0,0,np.pi/8, np.pi/8, np.pi/8, 0,0))

In [ ]:
import os
import glob
import numpy as np
from itertools import product

# Parameters
N = 14  # Number of qubits
T = 1000  # Time steps
L = 3  # Grid size
circuit_realizations = 10  # Number of circuit realizations

# Initialize empty arrays for magic and entanglement
magic_combined = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)
entanglement_combined = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)

# Base directory for results
base_dir = '../combined_results'

# Get Jx and Jz values
J_vals = np.linspace(0.1, np.pi - 0.1, L)

# Loop through all combinations of Jx, Jz and circuit realizations
for ix, jx in enumerate(J_vals):
    for iz, jz in enumerate(J_vals):
        for cr in range(circuit_realizations):
            # Construct paths to the individual CR files
            magic_file = os.path.join(base_dir, 'magic', f'N{N}', f'T{T}', f'Jx{jx:.2f}', f'Jz{jz:.2f}', f'cr{cr}.npz')
            entanglement_file = os.path.join(base_dir, 'entanglement', f'N{N}', f'T{T}', f'Jx{jx:.2f}', f'Jz{jz:.2f}', f'cr{cr}.npz')
            
            # Check if files exist and load them
            if os.path.exists(magic_file):
                magic_data = np.load(magic_file)
                # Individual file should have shape (T+1,)
                magic_combined[cr, ix, iz, :] = magic_data['magic']
                print(f"Loaded magic data for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}, shape: {magic_data['magic'].shape}")
            else:
                print(f"Warning: Magic data not found for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}")
                
            if os.path.exists(entanglement_file):
                entanglement_data = np.load(entanglement_file)
                # Individual file should have shape (T+1,)
                entanglement_combined[cr, ix, iz, :] = entanglement_data['entan']
                print(f"Loaded entanglement data for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}, shape: {entanglement_data['entan'].shape}")
            else:
                print(f"Warning: Entanglement data not found for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}")

# Now magic_combined and entanglement_combined have shape (circuit_realizations, L, L, T+1)
print(f"Magic combined shape: {magic_combined.shape}")
print(f"Entanglement combined shape: {entanglement_combined.shape}")

# You can save these combined arrays if needed
np.savez(
    f'../combined_matrices_N{N}_T{T}_L{L}_cr{circuit_realizations}.npz',
    magic=magic_combined,
    entan=entanglement_combined
)
print(f"Saved combined matrices to combined_matrices_N{N}_T{T}_L{L}_cr{circuit_realizations}.npz")

In [ ]:
import os
import glob
import numpy as np
from itertools import product

# Parameters
N = 14  # Number of qubits
T = 1000  # Time steps
L = 10  # Grid size
circuit_realizations = 10  # Number of circuit realizations

# Initialize empty arrays for magic and entanglement
magic_combined = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)
entanglement_combined = np.zeros((circuit_realizations, L, L, T + 1), dtype=np.float64)

# Base directory for combined results
base_dir = '../combined_results'

# Get Jx and Jz values
J_vals = np.linspace(0.1, np.pi - 0.1, L)

# Loop through all combinations of circuit realizations, Jx and Jz
for cr in range(circuit_realizations):
    for ix, jx in enumerate(J_vals):
        for iz, jz in enumerate(J_vals):
            # Construct file paths for the specific circuit realization, Jx, Jz combination
            magic_file = os.path.join(base_dir, 'magic', f'N{N}', f'T{T}', f'Jx{jx:.2f}', f'Jz{jz:.2f}', f'cr{cr}.npz')
            entanglement_file = os.path.join(base_dir, 'entanglement', f'N{N}', f'T{T}', f'Jx{jx:.2f}', f'Jz{jz:.2f}', f'cr{cr}.npz')
            
            # Check if magic file exists and load it
            if os.path.exists(magic_file):
                try:
                    magic_data = np.load(magic_file)
                    # The key is 'magic'
                    magic_combined[cr, ix, iz, :] = magic_data['magic']
                except Exception as e:
                    print(f"Error loading magic data for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}: {e}")
            else:
                print(f"Warning: Magic data not found for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}")
            
            # Check if entanglement file exists and load it
            if os.path.exists(entanglement_file):
                try:
                    entanglement_data = np.load(entanglement_file)
                    # Try both possible keys for entanglement
                    if 'entan' in entanglement_data:
                        entanglement_combined[cr, ix, iz, :] = entanglement_data['entan']
                    elif 'entanglement' in entanglement_data:
                        entanglement_combined[cr, ix, iz, :] = entanglement_data['entanglement']
                    else:
                        print(f"Warning: No valid key found in {entanglement_file}")
                        print(f"Available keys: {entanglement_data.files}")
                except Exception as e:
                    print(f"Error loading entanglement data for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}: {e}")
            else:
                print(f"Warning: Entanglement data not found for CR={cr}, Jx={jx:.2f}, Jz={jz:.2f}")

# Now magic_combined and entanglement_combined have shape (circuit_realizations, L, L, T+1)
print(f"Magic combined shape: {magic_combined.shape}")
print(f"Entanglement combined shape: {entanglement_combined.shape}")

# Check for missing data points
magic_missing = np.isclose(magic_combined, 0).all(axis=3).sum()
entanglement_missing = np.isclose(entanglement_combined, 0).all(axis=3).sum()
total_points = circuit_realizations * L * L

print(f"Missing magic data points: {magic_missing}/{total_points} ({magic_missing/total_points*100:.1f}%)")
print(f"Missing entanglement data points: {entanglement_missing}/{total_points} ({entanglement_missing/total_points*100:.1f}%)")

In [ ]:
magic_data['magic'].shape

### Check that gates match

In [ ]:
import functions as fn
import numpy as np
sx, sy, sz, id_ = fn.X, fn.Y, fn.Z, fn.I

def print_paulis(gave_evo):
    PAULI = {'Z': sz, 'Y': sy, '1': id_, 'X': sx, }
    for i in PAULI:
        pi = PAULI[i]
        for j in PAULI:
            pj = PAULI[j]; overlap = np.trace(gave_evo @ np.kron(pi, pj))
            if np.abs(overlap) > 1e-10:
                print(i,j,f' -> {overlap.real:.9f}', sep='')

In [ ]:
with open('../random_archive/angles_N2.json', 'r', encoding='utf-8') as f:
    loaded_dict = json.load(f)
params = loaded_dict[0][0]
Jz = params['Jz']; Jx = params['Jx']
θ1 = params['θ1']; θ2 = params['θ2']
θ3 = params['θ3']; θ4 = params['θ4']

ale = fn.gate_xyz_disordered(θ1, θ2, Jx/2, -Jx/2, Jz/2, θ3, θ4)
fn.print_matrix(ale)

gate = np.kron(sz, sx)/4
print(f'\n','~'*30,'\nbefore evolution:\n', sep='')
fn.print_matrix(gate, 2)
print_paulis(gate)
gave_evo = ale.conj().T @ gate @ ale
print(f'\n','~'*30,'\nafter evolution:\n', sep='')
fn.print_matrix(gave_evo, 2)
print_paulis(gave_evo)